# Calculate MAE between AEMET and datacube data:

$$
MAE = \frac{1}{n}\sum_{i=1}^{n} \lvert aemet\_val\_i - datacube\_val\_i\rvert
$$

For wind direction, the error is calculated as follows:

$$
MAE = \frac{1}{n}\sum_{i=1}^{n} \min(|aemet\_val\_i - datacube\_val\_i|, 360 - |aemet\_val\_i - datacube\_val\_i|)
$$


In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import pickle
import os
from tqdm import tqdm
from src.DatacubeValidation.stations import get_valid_aemet_stations_geodataframe, get_aemet_station_data_from_id, process_aemet_station_data
from src.config import DATACUBE_PATH, AEMET_VALIDATION_RESUTLS_DIR

2025-04-22 11:03:22.828 | INFO     | src.config:<module>:13 - PROJ_ROOT path is: C:\Workspace\Projects\IberFire


In [2]:
valid_stations = get_valid_aemet_stations_geodataframe()
datacube = xr.open_dataset(DATACUBE_PATH)

In [3]:
features_to_validate = ["TMAX", "TMIN", "TMEDIA", "PRECIPITACION", "DIR", "VELMEDIA", "RACHA", "PRESMAX", "PRESMIN"]
map_to_datacube_featurenames = {
    "TMAX": "t2m_max",
    "TMIN": "t2m_min",
    "TMEDIA": "t2m_mean",
    "PRECIPITACION": "total_precipitation_mean",
    "DIR": "wind_direction_at_max_speed",
    "VELMEDIA": "wind_speed_mean",
    "RACHA": "wind_speed_max",
    "PRESMAX": "surface_pressure_max",
    "PRESMIN": "surface_pressure_min"
}
for i, row in tqdm(valid_stations.iterrows()): # 816 valid stations
    if os.path.exists(os.path.join(AEMET_VALIDATION_RESUTLS_DIR, "stations_MAE", f"validation_MAE_{row['INDICATIVO']}.pkl")):
        continue
    else:
        validation_results = {} # To save the mean squared error for each feature

    id = row["INDICATIVO"]
    x_coord = row["X_COORDINATE"]
    y_coord = row["Y_COORDINATE"]
    # Clip datacube to the station coordinates
    small_datacube = datacube.sel(x=x_coord, y=y_coord, method="nearest")
    # Get the station data
    station_data = get_aemet_station_data_from_id(id)
    station_data = process_aemet_station_data(station_data)
    features_to_validate_available_in_station = station_data.columns.intersection(features_to_validate).tolist()

    if len(features_to_validate_available_in_station) == 0: 
        print(f"Station {id} has no features to validate")
        continue
    else:
        for feature in features_to_validate_available_in_station:
            validation_results[feature] = []

    # Compare the datacube values with the station data values
    for _, validation_day in station_data.iterrows():
        date = validation_day["FECHA"]
        for feature in features_to_validate_available_in_station:
            aemet_val = validation_day[feature]
            datacube_val = small_datacube[map_to_datacube_featurenames[feature]].sel(time=date).values

            if feature == "DIR":
                diff = abs(aemet_val - datacube_val)
                error = np.minimum(diff, 360 - diff)
                validation_results[feature].append(error)
            else:
                error = np.abs(aemet_val - datacube_val)
                validation_results[feature].append(error)

    for feature in features_to_validate_available_in_station:
        validation_results[feature] = np.nanmean(validation_results[feature])
    
    # Save the validation results to a pickle file
    with open(os.path.join(AEMET_VALIDATION_RESUTLS_DIR, "stations_MAE", f"validation_MAE_{row['INDICATIVO']}.pkl"), "wb") as f:
        pickle.dump(validation_results, f)


816it [00:00, 25495.58it/s]


# Calculate minimum and maximum values to normalize MAE:

$$
Normalized MAE = \frac{MAE}{y_{max} - y_{min}}
$$

In [4]:
minmax_dict = {}

for i,row in tqdm(valid_stations.iterrows()):
    id = row["INDICATIVO"]
    minmax_dict[id] = {}
    station_data = get_aemet_station_data_from_id(id)
    station_data = process_aemet_station_data(station_data)
    features_to_validate_available_in_station = station_data.columns.intersection(features_to_validate).tolist()
    for feature in features_to_validate_available_in_station:
        min_val = station_data[feature].min()
        max_val = station_data[feature].max()
        minmax_dict[id][f"{feature}_min"] = min_val
        minmax_dict[id][f"{feature}_max"] = max_val
    # Correct wind direction
    minmax_dict[id]["DIR_min"] = 0


# Save the min and max values to a pickle file
with open(os.path.join(AEMET_VALIDATION_RESUTLS_DIR, f"AEMET_minmax_dict.pkl"), "wb") as f:
    pickle.dump(minmax_dict, f)
    

816it [00:25, 31.98it/s]
